In [2]:
import sys

In [3]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase

In [4]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [5]:
uri = "bolt://neo4j:7687"
username = "neo4j"
password = "neo4jpassword"

In [8]:
# Jupyter docker내에서 억세스 할 수 있는 데이터 위치
HOME_DIR = "/app/data/rdb_reformulate"

### neo4j database 초기화
- 다른 필요한 데이터가 들어있을 경우 실행하면 안 됨

In [9]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
   # 모든 관계와 노드 삭제
   session.run("MATCH (n) DETACH DELETE n")

driver.close()

### ProductMeta 노드 생성 (root 노드)
- Label: DatabaseType
- Property
  - type: 'ProductMeta'

In [10]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
   session.run("CREATE (:DatabaseType {type: 'ProductMeta'})")

driver.close()

### 상위 카테고리 노드 생성
- Label: Category
- Property
  - name: 상위 category 이름
- Parent Relation
  - DatabaseType('ProductMeta') -[ HAS_category ]-> Category('...')

In [11]:
name_list = ['AutoProductChange', 'Benefit', 'Campaign', 'CommonRule', 'CustomerInfo', 'Data', 'OptionData', 'Product', 'SmsText', 'TopupInfo', 'Voice']
# name_list = [ "Benefit", "Product" ]

driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # Category 노드 생성
    for name in name_list:
        session.run(f"CREATE (:Category {{name: '{name}'}})")

    # ProductMeta 노드와 Category 노드 연결
    session.run("""
       MATCH (pm:DatabaseType {type: 'ProductMeta'})
       MATCH (c:Category)
       CREATE (pm)-[:HAS_Category]->(c)
    """)

driver.close()

### product > plan
- 모바일 요금제 상품
- Label: Plan
- Property
  - 요금제 속성들
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...')

In [25]:
meta_path = os.path.join(HOME_DIR, "product", "plan", "meta.csv")

In [26]:
meta_table = pd.read_csv(meta_path)

In [27]:
path = os.path.join(HOME_DIR, "product", "meta.csv")
product_meta = pd.read_csv(path)

In [28]:
meta_table_enriched = pd.merge(
    meta_table, 
    product_meta, 
    on='pmProductId', 
    how='left',
    suffixes=('', '_entire')
)

In [30]:
columns_to_drop = ['approvalinfo', 'versioninfo', 'type']

In [32]:
meta_table = meta_table_enriched.drop(columns=columns_to_drop)

KeyError: "['approvalinfo', 'versioninfo', 'type'] not found in axis"

In [ ]:
def type_cast(input_data):
    new_data = None
    if '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data = input_data.replace("nan", "")
    new_data = ast.literal_eval(input_data)

    return new_data

In [ ]:
# 리스트형으로 변환
meta_table["generation"] = meta_table["generation"].apply(lambda x: type_cast(x))
meta_table["marketingkeyword"] = meta_table["marketingkeyword"].apply(lambda x: type_cast(x))

In [ ]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table.iterrows():
        props = {}
        for col, val in row.items():
            if col in ('generation', 'marketingkeyword'):
                props[col] = list(val) if isinstance(val, (list, tuple)) else []
            else:
                # NaN/None 스킵
                if pd.isna(val):
                    continue
                # pandas dtype → Python 기본 타입
                if isinstance(val, (np.integer, int)):
                    props[col] = int(val)
                elif isinstance(val, (np.floating, float)):
                    props[col] = float(val)
                elif isinstance(val, (np.bool_, bool)):
                    props[col] = bool(val)
                else:
                    props[col] = str(val)

        session.run(
            "CREATE (n:Plan $props)",
            props=props
        )
driver.close()

In [ ]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    session.run("""
        MATCH (c:Category {name: 'Product'})
        MATCH (pl:Plan)
        CREATE (c)-[:HAS_Plan]->(pl)
    """)

driver.close()

### autoProductChange
- 요금제 자동 변경 정책
- Label: X (relation only)
- Property
  - 변경 조건들 (changerule_base, changerule_date)
- Parent Relation
  - Product('...') -[relationType 칼럼 값]-> Product('...')

In [33]:
meta_path = os.path.join(HOME_DIR, "autoProductChange", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [34]:
meta_table = meta_table.rename(columns={
    "autoproductchange|changerule|datebase|value": "changerule_base",
    "autoproductchange|changerule|date|value": "changerule_date"
})

In [35]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table.iterrows():
        src = row["pmProductId"]
        tgt = row["targetProductId"]
        base = row["changerule_base"]
        date = row["changerule_date"]
        rel_type = row["relationType"]

        cypher = f"""
        MATCH (a:Plan {{pmProductId: $src}}), (b:Plan {{pmProductId: $tgt}})
        CREATE (a)-[r:{rel_type} {{
            changerule_base: $base,
            changerule_date: $date
        }}]->(b)
        """
        session.run(cypher, src=src, tgt=tgt, base=base, date=date)

driver.close()

In [36]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table.iterrows():
        src = row["pmProductId"]
        tgt = row["targetProductId"]
        base = row["changerule_base"]
        date = row["changerule_date"]
        rel_type = row["relationType"]

        cypher = f"""
        MATCH (a:plan {{pmProductId: $src}}), (b:plan {{pmProductId: $tgt}})
        CREATE (a)-[r:{rel_type} {{
            changerule_base: $base,
            changerule_date: $date
        }}]->(b)
        """
        session.run(cypher, src=src, tgt=tgt, base=base, date=date)

driver.close()

### benefit > data
- 데이터 충전, 선물 혜택 관련 정보
- Label: 테이블의 각 칼럼을 label로 분해 (그래프의 관점에서 이렇게 구성하는것이 더 타당해 보여서)
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Benefit') -[HAS_SubCategory]-> Category('Benefit_Data') -[CONTAINS]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [37]:
meta_path = os.path.join(HOME_DIR, "benefit", "data", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [38]:
meta_table.head(5)

,pmProductId,datagiftreceivingavailability,maximumshareamount,datarefillamount,datarefillcoupongiftingavailability
0,PA00000001,True,2.0,15.0,True
1,PA00000002,True,2.0,1.8,True
2,PA00000003,True,2.0,5.0,True
3,PA00000004,True,2.0,30.0,True
4,PA00000005,True,2.0,54.0,True


In [39]:
cols = [
    "datagiftreceivingavailability",
    "maximumshareamount",
    "datarefillamount",
    "datarefillcoupongiftingavailability"
]

In [40]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # benefit_data 노드 하나 생성하고 Benefit 노드와 연결 (그냥 data로 하면 다른 레이블과 혼동됨)
    session.run("MERGE (:Category {name:'Benefit_Data'})")
    session.run("""
        MATCH (b:Category {name: 'Benefit'})
        MATCH (bd:Category {name:'Benefit_Data'})
        MERGE (b)-[:HAS_SubCategory]->(bd)
    """)
driver.close()

In [41]:
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    for col in cols:
        # 칼럼명을 레이블로 하고, 해당 칼럼의 유니크한 값들을 각각 노드로 생성
        for val in meta_table[col].dropna().unique():
            session.run(
                f"MERGE (n:`{col}` {{ value: $val }})",
                val=val
            )

    # 상품과 관계 생성
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue

            session.run(
                f"""
                MATCH (p:Plan {{pmProductId: $pm}})
                MATCH (v:`{col}`    {{ value: $val }})
                MERGE (p)-[:HAS_{col}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()

In [42]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:  
    
    for record in meta_table.to_dict("records"):
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue
            session.run(
                f"""
                MATCH (b:Category {{name:'Benefit_Data'}})
                MATCH (n:{col})
                MERGE (b)-[:CONTAINS]->(n)
                """
            )
driver.close()

### benefit > product
- 미처리

### benefit > voice
- 통화 충전 관련 정보
- Label
  - voicecallrefill
- Property
  - refillamount
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Benefit') -[HAS_SubCategory]-> Category('Benefit_Voice') -[CONTAINS]-> voicecallrefill('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_voicecallrefill(voicecallrefillrange:[...])]-> 칼럼명('...')

In [43]:
meta_path = os.path.join(HOME_DIR, "benefit", "voice", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [44]:
meta_table["voicecallrefillrange"] = meta_table["voicecallrefillrange|range"].apply(lambda x: ast.literal_eval(x))

In [45]:
del meta_table["voicecallrefillrange|range"]

In [46]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # benefit_voice 노드 하나 생성하고 Benefit 노드와 연결 (그냥 voice로 하면 다른 레이블과 혼동됨)
    session.run("MERGE (:Category {name:'Benefit_Voice'})")
    session.run("""
        MATCH (b:Category {name: 'Benefit'})
        MATCH (bv:Category {name:'Benefit_Voice'})
        MERGE (b)-[:HAS_SubCategory]->(bv)
    """)
driver.close()

In [47]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for row in meta_table.to_dict("records"):
        pm_id   = row["pmProductId"]
        ranges  = row["voicecallrefillrange"]    # 리스트 타입
        refill  = float(row["refillamount"])     # float
    
        session.run(
                """
                MATCH (p:Plan {pmProductId: $pm})
                MERGE (v:voicecallrefill {refillamount: $refill})
                CREATE (p)-[:HAS_voicecallrefill {
                    voicecallrefillrange: $ranges
                }]->(v)
                """,
                pm=pm_id,
                refill=refill,
                ranges=ranges
            )

driver.close()

In [48]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:  
    session.run(
        f"""
        MATCH (b:Category {{name:'Benefit_Voice'}})
        MATCH (n:voicecallrefill)
        MERGE (b)-[:CONTAINS]->(n)
        """
    )
driver.close()

In [49]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for row in meta_table.to_dict("records"):
        pm_id   = row["pmProductId"]
        ranges  = row["voicecallrefillrange"]    # 리스트 타입
        refill  = float(row["refillamount"])     # float

        session.run(
            """
            MATCH (p:plan {pmProductId: $pm})
            MERGE (v:voicecallrefill {refillamount: $refill})
            CREATE (p)-[:HAS_voicecallrefill {
                voicecallrefillrange: $ranges
            }]->(v)
            """,
            pm=pm_id,
            refill=refill,
            ranges=ranges
        )

driver.close()

In [50]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    session.run(
        """
        MATCH (b:benefit_voice)
        MATCH (n:voicecallrefill)
        MERGE (b)-[:HAS_benefit]->(n)
        """, val=val
    )
driver.close()

### campaign > targetproduct
- 각각의 캠페인을 별도의 노드로 생성
- Label
  - Campaign
- Property
  - productName, legacyProductId
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Campaign') -[HAS_Campaign]-> Campaign('...')

In [51]:
meta_path = os.path.join(HOME_DIR, "campaign", "targetproduct.csv")
meta_table = pd.read_csv(meta_path)

In [52]:
meta_table.head(5)

,pmProductId,productName,legacyProductId
0,BB00000001,T공시지원금,NaN
1,BA00000031,선택약정,NaN
2,BA00000028,0히어로 혜택,NaN
3,BA00000073,요금약정할인제도,NaN
4,BA00000047,Wavve 2천원 할인,NaN


In [53]:
records = meta_table.to_dict("records")

driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    session.run(
        """
        UNWIND $rows AS row
        MERGE (c:Campaign {pmProductId: row.pmProductId})
        SET c.productName      = row.productName,
            c.legacyProductId = row.legacyProductId
        WITH c
        MATCH (category:Category {name: 'Campaign'})
        MERGE (category)-[:HAS_Campaign]->(c)
        """,
        rows=records
    )
driver.close()

### campaign > relation
- 위의 캠페인을 요금제 상품과 연결
- relation의 property를 relationshipType으로 정의

In [54]:
meta_path = os.path.join(HOME_DIR, "campaign", "relation.csv")
meta_table = pd.read_csv(meta_path)

In [55]:
meta_table.head(5)

,pmProductId,targetProductId,relationshipType
0,PA00000001,BB00000001,signupPreTermination
1,PA00000001,BA00000031,signupConcurrentTermination
2,PA00000001,BA00000028,terminationPreTermination
3,PA00000001,BA00000073,terminationPreTermination
4,PA00000001,BA00000047,terminationConcurrentTermination


In [56]:
records = meta_table.to_dict("records")

In [57]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    session.run(
        """
        UNWIND $rows AS row
        MATCH (p:Plan     {pmProductId: row.pmProductId})
        MATCH (c:Campaign {pmProductId: row.targetProductId})
        MERGE (p)-[r:HAS_signupcondition]->(c)
        SET r.relationshipType = row.relationshipType
        """,
        rows=records
    )

driver.close()

In [58]:
meta_path = os.path.join(HOME_DIR, "campaign", "relation.csv")
meta_table = pd.read_csv(meta_path)

In [59]:
records = meta_table.to_dict("records")
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    session.run(
        """
        UNWIND $rows AS row
        MATCH (p:plan     {pmProductId: row.pmProductId})
        MATCH (c:campaign {pmProductId: row.targetProductId})
        MERGE (p)-[r:HAS_signupcondition]->(c)
        SET r.relationshipType = row.relationshipType
        """,
        rows=records
    )

driver.close()

### commonRule
- 미처리

### customerInfo
- 가입조건에 대한 테이블, 일단 이 중에서 agerule만 사용
- Label: MinAge, MaxAge
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('CustomerInfo') -[CONTAINS]-> MinAge('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_MinAge]-> MinAge('...')

In [60]:
meta_path = os.path.join(HOME_DIR, "customerinfo", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [61]:
meta_table.head(5)

,pmProductId,welfarediscounttypeavailability,agerule,businesscustomersubtyperule,customertyperule,individualcustomersubtyperule,organizationcustomersubtyperule
0,PA00000001,{},"[0, 999]","['SKBroadband㈜', 'SKTelecom㈜']",['개인'],"['공무원', '일반']",{}
1,PA00000002,{},"[0, 999]","['SKBroadband㈜', 'SKTelecom㈜']",['개인'],"['공무원', '일반']",{}
2,PA00000003,{},"[0, 999]","['SKBroadband㈜', 'SKTelecom㈜']",['개인'],"['공무원', '일반']",{}
3,PA00000004,{},"[0, 999]","['SKBroadband㈜', 'SKTelecom㈜']",['개인'],"['공무원', '일반']",{}
4,PA00000005,{},"[0, 999]","['SKBroadband㈜', 'SKTelecom㈜']",['개인'],"['공무원', '일반']",{}


In [62]:
meta_table["agerule"] = meta_table["agerule"].apply(lambda x: [ int(y) for y in ast.literal_eval(x) ])

In [63]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for row in meta_table.to_dict("records"):
        pm_id = row["pmProductId"]
        age_range = row["agerule"]
        min_age = min(age_range)
        max_age = max(age_range)
    
        session.run(
                """
                MATCH (p:Plan {pmProductId: $pm})
                MATCH (c:Category {name: 'CustomerInfo'})
                MERGE (ma:MinAge {value: $min_age})
                MERGE (xa:MaxAge {value: $max_age})
                MERGE (p)-[:HAS_minAge]->(ma)
                MERGE (p)-[:HAS_maxAge]->(xa)
                MERGE (c)-[:CONTAINS]->(ma)
                MERGE (c)-[:CONTAINS]->(xa)
                """,
                pm=pm_id,
                min_age = min_age,
                max_age = max_age
            )

driver.close()

### data
- 요금제 별 데이터 정책
- Label: 각 칼럼을 label로 하고, 각 칼럼의 유니크한 값들을 노드로 정의
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Data') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [64]:
meta_path = os.path.join(HOME_DIR, "data", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [65]:
meta_table

,pmProductId,includeddataforsharingandtethering,includeddataseparatesetting,includedmvoip,appliedspeed,generaldataexceedlimit|availabletoapply,includeddata,seniordataexceedlimit|availabletoapply
0,PA00000001,15.0,0,15.0,1024,False,15.0,False
1,PA00000002,1.8,0,1.8,0,True,1.8,False
2,PA00000003,5.0,0,5.0,1024,False,5.0,False
3,PA00000004,30.0,0,100.0,5120,False,100.0,False
4,PA00000005,54.0,0,200.0,5120,False,200.0,False
...,...,...,...,...,...,...,...,...
120,PA00002811,120.0,0,999999.0,0,False,999999.0,False
121,PA00002812,120.0,0,999999.0,0,False,999999.0,False
122,PA00002813,100.0,0,999999.0,0,False,999999.0,False
123,PA00002814,80.0,0,999999.0,0,False,999999.0,False


In [66]:
for col in meta_table.columns:
    print(col, meta_table[col].nunique())

pmProductId 125
includeddataforsharingandtethering 36
includeddataseparatesetting 1
includedmvoip 44
appliedspeed 5
generaldataexceedlimit|availabletoapply 2
includeddata 44
seniordataexceedlimit|availabletoapply 2


In [67]:
col_rename = {
    "generaldataexceedlimit|availabletoapply":  "generaldataexceedlimit",
    "seniordataexceedlimit|availabletoapply":  "seniordataexceedlimit"
}

In [68]:
meta_table = meta_table.rename(columns=col_rename)

In [69]:
cols = [
    "includeddataforsharingandtethering",
    "includeddataseparatesetting",
    "includedmvoip",
    "appliedspeed",
    "generaldataexceedlimit",
    "includeddata",
    "seniordataexceedlimit"
]

In [70]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        for val in meta_table[col].dropna().unique():
            session.run(
                f"MERGE (n:`{col}` {{ value: $val }})",
                val=val
            )

    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue
            session.run(
                f"""
                MATCH (p:plan {{pmProductId: $pm}})
                MATCH (v:`{col}` {{ value: $val }})
                MERGE (p)-[:HAS_{col}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()

In [71]:
meta_table.head(5)

,pmProductId,includeddataforsharingandtethering,includeddataseparatesetting,includedmvoip,appliedspeed,generaldataexceedlimit,includeddata,seniordataexceedlimit
0,PA00000001,15.0,0,15.0,1024,False,15.0,False
1,PA00000002,1.8,0,1.8,0,True,1.8,False
2,PA00000003,5.0,0,5.0,1024,False,5.0,False
3,PA00000004,30.0,0,100.0,5120,False,100.0,False
4,PA00000005,54.0,0,200.0,5120,False,200.0,False


In [72]:
col_rename = {
    "generaldataexceedlimit|availabletoapply":  "generaldataexceedlimit",
    "seniordataexceedlimit|availabletoapply":  "seniordataexceedlimit"
}
meta_table = meta_table.rename(columns=col_rename)

In [73]:
cols = [
    "includeddataforsharingandtethering",
    "includeddataseparatesetting",
    "includedmvoip",
    "appliedspeed",
    "generaldataexceedlimit",
    "includeddata",
    "seniordataexceedlimit"
]

In [74]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        # 해당 컬럼의 유니크 값만 뽑아서 노드 생성
        for val in meta_table[col].dropna().unique():
            session.run(
                f"""
                MATCH (c:Category {{ name: 'Data' }})
                MERGE (n:`{col}` {{ value: $val }})
                MERGE (c)-[:CONTAINS]->(n)
                """,
                val=val
            )
    
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue
            session.run(
                f"""
                MATCH (p:Plan {{pmProductId: $pm}})
                MATCH (v:`{col}` {{ value: $val }})
                MERGE (p)-[:HAS_{col}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()    
    

### optionData
- 미처리

### product > otherproduct
- 미처리 (비어있음)

### smstext
- Label: 각 칼럼
- Property
  - 각 칼럼의 유니크한 값
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('SmsText') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [75]:
meta_path = os.path.join(HOME_DIR, "smstext", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [76]:
meta_table.head(5)

,pmProductId,textrange,includedtext
0,PA00000001,0,999999.0
1,PA00000002,0,999999.0
2,PA00000003,0,999999.0
3,PA00000004,0,999999.0
4,PA00000005,0,999999.0


In [77]:
cols = [ "textrange", "includedtext" ]

In [78]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        for val in meta_table[col].dropna().unique():
            session.run(
                f"""
                MATCH (c:Category {{ name: 'SmsText' }})
                MERGE (n:{col} {{ value: $val }})
                MERGE (c)-[:CONTAINS]->(n)
                """,
                val=val
            )

    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]

            session.run(
                f"""
                MATCH (p:Plan {{ pmProductId: $pm }}),
                      (v:{col} {{ value: $val }})
                MERGE (p)-[:HAS_{col}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()

### topupinfo
- 미처리

### voice
- voicecallrange|providingamount와 includedvoicecall의 차이를 모르겠음
- includedvideoorvalueaddedcall, voicecallrange|providingamount, includedvoicecall만 사용
- Label: 각 칼럼
- Property
  - 각 칼럼의 유니크한 값
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Voice') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [79]:
meta_path = os.path.join(HOME_DIR, "voice", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [80]:
meta_table = meta_table.rename({
    "voicecallrange|providingamount": "voicecallrange_providingamount",
}, axis=1)

In [81]:
meta_table = meta_table[["pmProductId", "includedvideoorvalueaddedcall", "voicecallrange_providingamount", "includedvoicecall"]].copy()

In [82]:
meta_table.head(5)

,pmProductId,includedvideoorvalueaddedcall,voicecallrange_providingamount,includedvoicecall
0,PA00000001,300,0,999999
1,PA00000002,100,0,999999
2,PA00000003,300,0,999999
3,PA00000004,300,0,999999
4,PA00000005,300,0,999999


In [83]:
cols = [
    "includedvideoorvalueaddedcall",
    "voicecallrange_providingamount",
    "includedvoicecall"
]

In [84]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        for val in meta_table[col].dropna().unique():
            session.run(
                f"""
                MATCH (c:Category {{ name: 'Voice' }})
                MERGE (n:`{col}` {{ value: $val }})
                MERGE (c)-[:CONTAINS]->(n)
                """,
                val=val
            )
            
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]

            session.run(
                f"""
                MATCH (p:Plan {{pmProductId: $pm}})
                MATCH (v:`{col}` {{value: $val}})
                MERGE (p)-[:HAS_{col}]->(v)
                """,
                pm=pm,
                val=val
            )

driver.close()